# 1. Import Libraries

In [1]:
import sys
from pathlib import Path

# Thêm thư mục gốc (project root) vào Python path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import xgboost as xgb

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error

import numpy as np

from utils.features import build_features

import optuna


# 2. Load Dataset

In [2]:
df_train, df_val = build_features(
                                df_train = pd.read_excel("../Dataset/training.xlsx"), 
                                df_test = None,
                                pls_fit_years = [1]
                                   )

In [4]:
features = [
    'Year', 'Month', 'Day', 'Hour',
    'Combined_Temp',
    'Combined_GHI_1', 'Combined_GHI_2'
]
target = 'Load'

# 3. Split Data

In [7]:
# Year 1 → TRAIN
train_mask = df_train['Year'] == 1

# Year 2 → VALIDATION
val_mask = df_val['Year'] == 2

# Year 3 → TEST
# Testing.xlsx is assumed to contain Year 3

X_train = df_train.loc[train_mask, features]
y_train = df_train.loc[train_mask, target]

X_val = df_val.loc[val_mask, features]
y_val = df_val.loc[val_mask, target]

print("Dataset sizes:")
print(f"Train (Year 1): {len(X_train):,}")
print(f"Validation (Year 2): {len(X_val):,}")

Dataset sizes:
Train (Year 1): 8,784
Validation (Year 2): 8,760


# 4. Train on year 1, evaluate on year 2

In [8]:
tscv = TimeSeriesSplit(n_splits=4)

In [9]:
def objective(trial):

    params = {
        "objective": "reg:squarederror",

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            1500
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            10
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-3,
            10,
            log=True
        ),

        "random_state": 42,
        "n_jobs": -1
    }

    fold_rmse = []

    # ============================================
    # Time-series CV trên YEAR 1
    # ============================================

    for train_idx, val_idx in tscv.split(X_train):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = xgb.XGBRegressor(**params)

        model.fit(
            X_fold_train,
            y_fold_train
        )

        y_pred = model.predict(
            X_fold_val
        )

        rmse = root_mean_squared_error(
            y_fold_val,
            y_pred
        )

        fold_rmse.append(rmse)

    # Optuna minimize mean CV RMSE
    return np.mean(fold_rmse)

In [10]:
study = optuna.create_study(
    direction="minimize",
    study_name="xgb_year1_timeseries_cv"
)

study.optimize(
    objective,
    n_trials=100,
    show_progress_bar=True
)

[I 2026-10-02 18:22:10,903] A new study created in memory with name: xgb_year1_timeseries_cv


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-10-02 18:22:13,110] Trial 0 finished with value: 315.3281440734863 and parameters: {'n_estimators': 792, 'learning_rate': 0.1739368002405972, 'max_depth': 8, 'min_child_weight': 9, 'subsample': 0.9875056549195667, 'colsample_bytree': 0.6497739720188765, 'gamma': 4.839028484332111, 'reg_alpha': 3.5136077434827067e-06, 'reg_lambda': 0.001652822117935116}. Best is trial 0 with value: 315.3281440734863.
[I 2026-10-02 18:22:15,628] Trial 1 finished with value: 305.4420356750488 and parameters: {'n_estimators': 982, 'learning_rate': 0.07820007940661113, 'max_depth': 7, 'min_child_weight': 2, 'subsample': 0.8127533564499771, 'colsample_bytree': 0.7161935159625922, 'gamma': 1.1750073762573832, 'reg_alpha': 1.0471786591448385e-06, 'reg_lambda': 0.034933418655702556}. Best is trial 1 with value: 305.4420356750488.
[I 2026-10-02 18:22:17,363] Trial 2 finished with value: 301.45520782470703 and parameters: {'n_estimators': 907, 'learning_rate': 0.03557411148133636, 'max_depth': 6, 'min_chi

In [11]:
print("Best CV RMSE:")
print(study.best_value)

print("\nBest parameters:")

for param, value in study.best_params.items():
    print(f"{param}: {value}")

Best CV RMSE:
268.64988708496094

Best parameters:
n_estimators: 735
learning_rate: 0.044974970480514756
max_depth: 3
min_child_weight: 6
subsample: 0.9726604916803252
colsample_bytree: 0.6624985904843348
gamma: 4.742422084291189
reg_alpha: 4.301446920624345e-07
reg_lambda: 0.01813392065633835


In [12]:
best_params = study.best_params

seeds = [0, 1, 2, 3, 42]

results = []

for seed in seeds:

    model = xgb.XGBRegressor(
        **best_params,
        objective="reg:squarederror",
        random_state=seed,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    y_pred = model.predict(X_val)

    results.append({
        "Seed": seed,
        "RMSE": root_mean_squared_error(y_val, y_pred),
        "MSE": mean_squared_error(y_val, y_pred),
        "MAE": mean_absolute_error(y_val, y_pred),
        "MAPE": mean_absolute_percentage_error(y_val, y_pred),
        "R2": r2_score(y_val, y_pred)
    })

stability_df = pd.DataFrame(results)

stability_summary = stability_df.drop(
    columns="Seed"
).agg(["mean", "std", "min", "max"]).T

print("=== Results by seed ===")
display(stability_df)

print("\n=== Stability summary ===")
display(stability_summary)

=== Results by seed ===


,Seed,RMSE,MSE,MAE,MAPE,R2
0,0,191.021301,36489.136719,140.947617,0.067398,0.779133
1,1,192.511795,37060.792969,141.847275,0.067816,0.775673
2,2,191.875931,36816.371094,141.244934,0.067498,0.777152
3,3,191.814819,36792.925781,141.405273,0.067595,0.777294
4,42,192.216568,36947.210938,141.718582,0.067745,0.776360



=== Stability summary ===


,mean,std,min,max
RMSE,191.888083,0.559854,191.021301,192.511795
MSE,36821.287500,214.692968,36489.136719,37060.792969
MAE,141.432736,0.362255,140.947617,141.847275
MAPE,0.067610,0.000172,0.067398,0.067816
R2,0.777122,0.001300,0.775673,0.779133


In [13]:
stability_df.to_csv("../report/seed_pls_val_report.csv", index = False)
stability_summary.to_csv("../report/seed_summary_pls_val_report.csv", index = False)

# 4. Final Training on year 1 and year 2

In [14]:
df_train, df_test = build_features(
                                df_train = pd.read_excel("../Dataset/training.xlsx"), 
                                df_test = pd.read_excel("../Dataset/testing.xlsx")
                                   )

In [15]:
X_train = df_train[features]
y_train = df_train[target]

X_test = df_test[features]

In [16]:
print("Dataset sizes:")
print(f"Train (Year 1): {len(X_train):,}")
print(f"Validation (Year 2): {len(X_val):,}")
print(f"Test (Year 3): {len(X_test):,}")

Dataset sizes:
Train (Year 1): 17,544
Validation (Year 2): 8,760
Test (Year 3): 8,760


In [19]:
seeds = [0, 1, 2, 3, 42]

test_predictions = []

for seed in seeds:

    model = xgb.XGBRegressor(
        **best_params,
        objective="reg:squarederror",
        random_state=seed,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(X_test)

    test_predictions.append(pred)

y_test_pred = np.mean(
    test_predictions,
    axis=0
)

In [20]:
pd.DataFrame(y_test_pred).to_csv("../report/pls_test_prediction.csv", index = False)